# Spatial Data Science with an LoD2 city model

The purpose of this Notebook is to illustrate a CityJSON-based spatial data science workflow with a simplified LoD2 city model.

In [1]:
#load the magic

%matplotlib inline
import os
import sys
from pathlib import Path
import webbrowser

import numpy as np
import pandas as pd
import shapely
from shapely.geometry import Polygon, shape, mapping
import json
import geojson

from osgeo import gdal, osr
osr.UseExceptions()

import matplotlib.pyplot as plt

In [2]:
import LoD2city3D

from cjio import cityjson

In [3]:
jparams = json.load(open('./param/uEstate_param.json'))   

In [4]:
# With the new 0.10.x syntax:
with open(jparams['cjsn_out'], 'r') as f:
    cm = cityjson.reader(f)

In [5]:
print(cm)

CityJSON version = 2.0.0
EPSG = 32734
bbox = [ 263778.110 6241078.022 28.800 265430.307 6242637.108 193.800 ]
Extensions = ['Energy']
=== CityObjects ===
|-- TINRelief (1)
|-- Road (58)
|-- Building (308)
    |-- BuildingInstallation (145)
        |-- +Energy-PVSystem (145)
|-- LandUse (3)
|-- SolitaryVegetationObject (11)
materials = False
textures = False


In [6]:
#- harvest the crs
theinfo = cm.get_info()
crs = theinfo[1]
print(crs)

EPSG = 32734


In [7]:
# 1. Grab the objects directly from the reader
buildings = {
    obj_id: obj 
    for obj_id, obj in cm.j['CityObjects'].items() 
    if obj.get('type') == 'Building'
}
vertices = cm.j['vertices']

In [8]:
vertices[:3]

[[-415067, -303525, 96400],
 [-418997, -299139, 96800],
 [-426038, -309401, 98200]]

In [9]:
transform_meta = cm.j.get('transform', {})
transform_meta

{'scale': [0.001, 0.001, 0.001],
 'translate': [264604.208314, 6241857.564832, 28.8]}

In [10]:
# 2. Extract surfaces with the transformation metadata included
geojson_output = LoD2city3D.extract_lod_surfaces(
    buildings, 
    vertices, 
    transform_meta=transform_meta, 
    crs_name=crs[7:]#"EPSG:32734"
)

# 3. Instantiate your clean GeoDataFrameLite directly from the output collection payload
gdf = LoD2city3D.GeoDataFrameLite.from_json(geojson_output)
gdf.crs = crs[7:]#"EPSG:32734"

In [11]:
#gdf.loc[1914].geometry
gdf.surface_type.unique()

array(['WallSurface', 'RoofSurface', 'GroundSurface'], dtype=object)

In [12]:
gdf.lod.unique()

array(['1.0', '2.0'], dtype=object)

In [13]:
gdf.tail(2)

,building_id,lod,surface_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry
4107,osm_1469054027,1.0,GroundSurface,185.659315,0.0,0.0,85.7,1469054027,66 Selbourne Road University Estate Cape Town,house,2.0,NaN,6.9,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264865.929, 6241...","POLYGON Z ((264873.028314 6241703.458832 85.7,..."
4108,osm_1469054027,1.0,GroundSurface,185.659314,0.0,0.0,78.8,1469054027,66 Selbourne Road University Estate Cape Town,house,2.0,NaN,6.9,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264865.929, 6241...","POLYGON Z ((264865.929314 6241707.344832 78.8,..."


In [14]:
unique_building_count = gdf['building_id'].nunique()
print(f"Total number of unique buildings: {unique_building_count}")

Total number of unique buildings: 308


<div class="alert alert-block alert-info"><b>Craveat</b> 
    
This community *(Woodstock, Salt River and Observatory)* is the second oldest community in South Africa. The buildings are old. Many have been repurposed. To account for refurbishment *--be as representative as possible--* and conform to the **[OpenStreetMap Guide](https://wiki.openstreetmap.org/wiki/Beginners%27_guide)** we typically tag these:  
`building=*` *~ the original purpose* `+` `building:use=*` *~ the current use*.

Furthermore; tagging in this community identifies **social housing, social facilities** (care home, shelter, etc.) **and informal housing** (backyard dwelling, shack, etc.) as `building / :use=residential`. **Student accomodation** includes the `residential=student` tag
</div>

In [15]:
# --- Account for idiosyncratic mapping at the surface level ---
gdf2 = gdf.copy()

# 1. Update surface classification if a current 'building:use' is present
if 'attr_building:use' in gdf2.columns:
    # Locate active residential updates while ignoring NaN records safely
    use_mask = (gdf2['attr_building:use'] == 'residential') & gdf2['attr_building:use'].notna()
    gdf2.loc[use_mask, 'attr_building'] = gdf2.loc[use_mask, 'attr_building:use']

# 2. Update surface classification to show active student housing usage
if 'attr_residential' in gdf2.columns:
    # Locate explicit student housing tags
    student_mask = (gdf2['attr_residential'] == 'student') & gdf2['attr_residential'].notna()
    gdf2.loc[student_mask, 'attr_building'] = gdf2.loc[student_mask, 'attr_residential']

# 1. Deduplicate by building_id so each physical asset is only counted once
unique_buildings_df = gdf2.drop_duplicates(subset=['building_id'])

# --- Verify the updated semantic surface distribution across the community ---
print("Updated building attribute distribution across harvested surfaces:")
print(unique_buildings_df['attr_building'].value_counts())

Updated building attribute distribution across harvested surfaces:
attr_building
house        295
garage         7
yes            4
warehouse      1
office         1
Name: count, dtype: int64


## 1. Spatial Data Science

<div class="alert alert-block alert-warning"><b>We start with basic spatial analysis</b>  
    
     
- We'll [estimate the population](#Section1a), within our area of interest, and then  
- calculate the [Building Volume Per Capita (BVPC)](#Section1b).
</div>

While estimating population is well documented; recent investigations to **understand overcrowding** have led to newer measurements.  

The most noteable of these is **Building Volume Per Capita (BVPC)** [(Ghosh, T; et al. 2020)](https://www.researchgate.net/publication/343185735_Building_Volume_Per_Capita_BVPC_A_Spatially_Explicit_Measure_of_Inequality_Relevant_to_the_SDGs). BVPC is the cubic meters of building per person. **BVPC tells us how much space one person has per residential living unit** (a house / apartment / etc.). It is ***a proxy measure of economic inequality and a direct measure of housing inequality***.

BVPC builds on the work of [(Reddy, A and Leslie, T.F., 2013)](https://www.tandfonline.com/doi/abs/10.1080/02723638.2015.1060696?journalCode=rurb20) and attempts to integrate with several **[Sustainable Development Goals](https://sdgs.un.org/goals)** (most noteably: **[SDG 11: Developing sustainable cities and communities](https://sdgs.un.org/goals/goal11)**) and captures the average ***'living space'*** each person has in their home.

<div class="alert alert-block alert-info"><b>These analysis expect the user to have some basic knowledge about the environment under inquiry / investigation</b> </div>

In [16]:
#len(unique_buildings_df)

In [17]:
#unique_buildings_df['attr_building'].unique()

<div class="alert alert-block alert-success"><b>1.  a) Estimate Population:</b> 
    
_(with population growth rate and population projection possible too)_ </div>

In [18]:
#- some data wrangling
with pd.option_context("future.no_silent_downcasting", True):
    unique_buildings_df = unique_buildings_df.assign(**{
        col: pd.to_numeric(
            unique_buildings_df[col].fillna(0).infer_objects(copy=False), errors='coerce'
        )
        for col in ['attr_building:flats', 'attr_building:units', 'attr_beds', 'attr_rooms', 'attr_building:levels']
        if col in unique_buildings_df.columns
    })

unique_buildings_df.head(2)

,building_id,lod,surface_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry
0,osm_13705114,1.0,WallSurface,536.217387,90.0,204.05,73.05,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264318.134, 62420...",POLYGON Z ((264304.860314 6241972.752832 69.60...
12,osm_739615941,2.0,RoofSurface,6.031476,0.0,0.00,100.20,739615941,10 Rhodes Avenue University Estate Cape Town,house,2.0,NaN,6.9,101.0,93.3,4FRW3C6X+WRX,"[[[264275.999, 6241813.835], [264270.087, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...


**This area is urban with single and 2-storey level housing units. To estimate population is thus pretty straight forward.**

<div class="alert alert-block alert-info"><b>We start with local knowledge.</b></div>

**On average there are roughly `4` people per `building:house` in this area.**  

**[social housing](https://en.wikipedia.org/wiki/Public_housing)** is tagged `building:residential` with the `3` people per building or `building:flats * 3` if the building is an *apartment-type* complex

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    
We will execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [19]:
#- average number of residents per formal house
f_house = 4
#- average number of residents per informal structure
inf_structure = 3

<div class="alert alert-block alert-warning"><b></b>  
    
**Furthermore:**    
- [social housing](https://en.wikipedia.org/wiki/Public_housing) is tagged `building:residential` with the number of occupants iether the number of informal structure occupants or `building:flats * inf_structure`
- A `social_facility` (carehome, shelter, etc.) harvests the beds `key:value` pair.
- `building:apartments` harvests the `building:flats` `key:value` pair (the number of units) to calculate `*3` people per  
    - ***Student accomodation***:  
>    - University owed: is tagged `building:dormitory` with `residential:university` and harvests the `beds` or `rooms` *'key:value'* pair.
>    - Private for-profit: is tagged `building:residential` or `:dormitory` with `residential:student` and then harvests the `building:flats` or `:rooms` *'key:value'* pair *(the number of units)* to calculate `*1` people per apartment; if `level: > 1` else `*3` people in a house share.  

**The tagging scheme and numbers is based on *how your community is mapped* and local knowledge**
</div>

In [20]:
#- population estimation
gdf_pop = unique_buildings_df.copy()
c = gdf_pop.columns

def pop(row):
    #- formal house
    if row['attr_building'] == 'house' or row['attr_building'] == 'semidetached_house':
        return f_house
    if row['attr_building'] == 'terrace' or row['attr_building'] == 'terraced':
        if 'attr_building:units' in c and row['attr_building:units'] != 0:
            return row['attr_building:units'] * f_house
        else:
            f_house

    #- informal structure (shack)
    if row['attr_building'] == 'cabin':
        return inf_structure
        
    #- in this case social housing
    if row['attr_building'] == 'residential' and 'social_facility' in c and row['attr_social_facility'] is np.nan:
        if row['attr_building:levels'] > 1:
            if 'attr_rooms' in c and row['attr_rooms'] != 0:
                return row['attr_rooms']
            if 'attr_building:flats' in c and row['attr_building:flats'] != 0:
                return row['attr_building:flats'] * inf_structure
        else:
            return inf_structure
    #-- social facility [shelter / carehome]
    if row['attr_building'] == 'residential' and row['attr_social_facility'] is not np.nan:
        if 'attr_building:units' in c and row['attr_building:units'] != 0:
            return row['attr_building:units'] * inf_structure
        else: 
            return row['attr_beds']
                
    #- formal apartment
    if row['attr_building'] == 'apartments':
        return row['attr_building:flats'] * 3
        
    #- private student residence 
    if row['attr_building'] == 'student':
        if row['attr_building:levels'] > 1:
            return row['attr_building:flats']
        else:
            return 3
    # university owned student residence
    if row['attr_building'] == 'dormitory' and row['attr_residential'] == 'university':
        if row['attr_building:levels'] > 1:
            if 'attr_rooms' in c and row['attr_rooms'] != 0:
                return row['rooms']
            if 'attr_beds' in c and row['attr_beds'] != 0:
                return row['attr_beds']
        else:
            return 3

gdf_pop['pop'] = gdf_pop.apply(lambda x: pop(x), axis=1)

est_pop = int(gdf_pop['pop'].sum())
print('The estimated population is:', est_pop)

The estimated population is: 1180


**[Statistics South Africa (STATSA)](https://www.statssa.gov.za) does not typically release official statistics at a neighbourhood level but its possible to find dedicated and resourceful geospatial professionals, in your immediate environment, who are just *plodding-away* doing awesome and useful stuff. These numbers are based on disaggregated [Statistics South Africa (STATSA)](https://www.statssa.gov.za) [Census 2011](https://www.statssa.gov.za/?page_id=3839) and can be accessed [here](https://census2011.adrianfrith.com/place/199041033)** 

**University Estate 987** (6 577: salt river, 9 207: observatory and 12 656: woodstock). 

We can calculate the annual population growth rate using the formula for **[Annual population growth](https://databank.worldbank.org/metadataglossary/health-nutrition-and-population-statistics/series/SP.POP.GROW):**

$$r = \frac{\ln{[\frac{End Population}{Start Population}}]}{n} * 100 = \frac{\ln{[\frac{1 180}{987}}]}{12} * 100   = 1.49\%$$

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [21]:
#- previous population
start_population = 987

#- period in years from the previous census
years = 12

In [22]:
#-execute
r = (np.log(est_pop/start_population)/years) * 100
print('population growth rate of approximately:', round(r, 2), '%')

population growth rate of approximately: 1.49 %


To conclude; we can project into the future with a very basic formula to estimate the population _x_-years from now:  

$$p  = P_o * (1 + r)^{t} = p = 1180 * (1 + 0.0149)^{10}  = 1 367$$

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the variables in the _`cell`_ below** </div>

In [23]:
#- period in years from now
years = 10

In [24]:
#- account for non-residential areas without failure
#- helper function
def safe_population_estimate(est_pop, r, years):
    try:
        p = est_pop * (1 + (r / 100))**years
        return int(p)
    except Exception as e:
        print(f"Population estimate failed: {e}")
        return None  # keeps notebook running

#- execute function
p = safe_population_estimate(est_pop, r, years)

#- shows error and moves on
if p is not None:
    print(f"estimated population {years} years from now: {p}")

estimated population 10 years from now: 1367


<div class="alert alert-block alert-success"><b>1. b) Building Volume Per Capita (BVPC):</b>  
BVPC = total population of a community divided by sum of building volume</div>

In [25]:
#- the surfaces
#gdf.head(2)

In [26]:

# 1. Compute the true total roof area per building from your filtered roofs
total_roof_areas = gdf_pop[gdf_pop['surface_type'] == 'RoofSurface'].groupby('building_id')['3d_area'].sum().to_dict()

# 2. Compute the true ground footprint area for your floor adjustments
ground_footprints = gdf_pop[gdf_pop['surface_type'] == 'GroundSurface'].groupby('building_id')['3d_area'].sum().to_dict()

# 3. Create your unique building dataframe
gdf_pop = gdf_pop.drop_duplicates(subset=['building_id']).copy()

# 4. Map the true aggregated totals back so the rows contain real asset sizes
gdf_pop['total_roof_area'] = gdf_pop['building_id'].map(total_roof_areas).fillna(0.0)
gdf_pop['ground_footprint'] = gdf_pop['building_id'].map(ground_footprints).fillna(0.0)

In [27]:
gdf_pop.head(2)

,building_id,lod,surface_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,...,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry,pop,total_roof_area,ground_footprint
0,osm_13705114,1.0,WallSurface,536.217387,90.0,204.05,73.05,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,...,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264318.134, 62420...",POLYGON Z ((264304.860314 6241972.752832 69.60...,NaN,0.000000,0.0
12,osm_739615941,2.0,RoofSurface,6.031476,0.0,0.00,100.20,739615941,10 Rhodes Avenue University Estate Cape Town,house,...,NaN,6.9,101.0,93.3,4FRW3C6X+WRX,"[[[264275.999, 6241813.835], [264270.087, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...,4.0,6.031476,0.0


In [28]:
# --- CELL 28: CALCULATE NATIVE 3D VOLUMNS FROM ALL FACETS ---

def calculate_facet_signed_volume(geometry):
    if geometry is None or (hasattr(geometry, 'is_empty') and geometry.is_empty):
        return 0.0
    vertices_3d = np.array(geometry.exterior.coords)
    if len(vertices_3d) < 4:
        return 0.0
    face_volume = 0.0
    p0 = vertices_3d[0]
    for i in range(1, len(vertices_3d) - 2):
        p1 = vertices_3d[i]
        p2 = vertices_3d[i + 1]
        face_volume += np.dot(p0, np.cross(p1, p2)) / 6.0
    return face_volume

# 1. Calculate individual facet volume contributions on your master dataframe
gdf['face_signed_volume'] = gdf['geometry'].apply(calculate_facet_signed_volume)

# 2. Group by building and sum them up completely
building_volume_series = gdf.groupby('building_id')['face_signed_volume'].sum()
volume_lookup_df = building_volume_series.reset_index(name='volume_m3')
#gdf.head(2)

In [29]:

# 1. Isolate unique building asset attributes (dropping individual face dimensions)
surface_cols = ['geometry', 'surface_type', 'tilt_deg', 'aspect_deg', '3d_area']
unique_buildings = gdf.drop_duplicates(subset=['building_id']).drop(columns=[col for col in surface_cols if col in gdf.columns]).copy()

# 2. Merge the total compiled volumes back safely
unique_buildings = unique_buildings.merge(volume_lookup_df, on='building_id')

In [30]:
#- remove the volume of the ground floor (unoccupied) when building:levels > 7 [this is an arbitrary number based on local knowledge]
#- typically the space is reserved for some other function: retail, etc. 
GROUND_FLOOR_HEIGHT_M = 2.8
adjusted_volumes = []

for idx, row in unique_buildings.iterrows():
    current_vol = row['volume_m3']
    levels = row.get('attr_building:levels', 0)
    levels = float(levels) if pd.notna(levels) and str(levels).strip() != '' else 0.0
    
    has_no_social_facility = ('attr_social_facility' not in unique_buildings.columns or pd.isna(row.get('attr_social_facility')))
    is_residential_target = str(row.get('attr_building')).lower() in ['house', 'semidetached_house', 'terrace', 'terraced', 'apartments']

    # FIX: Use 'ground_footprint' mapped from gdf_pop instead of the single facet '3d_area'
    if is_residential_target and levels > 7 and has_no_social_facility:
        b_id = row['building_id']
        # Grab the consolidated ground footprint we mapped to gdf_pop in Cell 26
        footprint_area = gdf_pop.loc[gdf_pop['building_id'] == b_id, 'ground_footprint'].values[0]
        deducted_vol = current_vol - (footprint_area * GROUND_FLOOR_HEIGHT_M)
        adjusted_volumes.append(max(0.0, deducted_vol))
    else:
        adjusted_volumes.append(current_vol)

# Assign finalized metrics back to your population summary dataframe
gdf_pop['volume_m3'] = adjusted_volumes
gdf_pop.head(2)

,building_id,lod,surface_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,...,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry,pop,total_roof_area,ground_footprint,volume_m3
0,osm_13705114,1.0,WallSurface,536.217387,90.0,204.05,73.05,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,...,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264318.134, 62420...",POLYGON Z ((264304.860314 6241972.752832 69.60...,NaN,0.000000,0.0,16778.099256
12,osm_739615941,2.0,RoofSurface,6.031476,0.0,0.00,100.20,739615941,10 Rhodes Avenue University Estate Cape Town,house,...,6.9,101.0,93.3,4FRW3C6X+WRX,"[[[264275.999, 6241813.835], [264270.087, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...,4.0,6.031476,0.0,1013.307644


In [31]:
gdf_pop['bvpc'] = np.where(
    gdf_pop['pop'] > 0,
    gdf_pop['volume_m3'] / gdf_pop['pop'],
    np.nan
)

gdf_pop.tail(2)

,building_id,lod,surface_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,...,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry,pop,total_roof_area,ground_footprint,volume_m3,bvpc
4093,osm_1459039139,1.0,WallSurface,25.007454,90.0,298.60,93.95,1459039139,NaN,garage,...,96.3,91.9,4FRW3F62+9H4,"[[[264446.096, 6241675.189], [264454.731, 6241...",POLYGON Z ((264446.09631399997 6241675.188832 ...,NaN,0.0,0.0,245.504498,NaN
4099,osm_1469054027,1.0,WallSurface,91.914369,90.0,118.94,82.25,1469054027,66 Selbourne Road University Estate Cape Town,house,...,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264865.929, 6241...","POLYGON Z ((264873.028314 6241703.458832 78.8,...",4.0,0.0,0.0,1281.049377,320.262344


In [32]:
print(gdf_pop['bvpc'].describe())

count    295.000000
mean     222.166349
std      125.647974
min       34.487385
25%      136.930773
50%      189.916061
75%      272.824098
max      971.828275
Name: bvpc, dtype: float64


In [33]:
bvpc = round(gdf_pop['volume_m3'].sum() / est_pop, 3)

print('Building Volume Per Capita (BVPC):', bvpc)

Building Volume Per Capita (BVPC): 251.452


<div class="alert alert-block alert-info"><b></b>

**This BVPC value is for all buildings with a `population > 0`. Buildings people live in** *(homes)*. 

And we can seperate `building:house` from `building:cabin` and `building:residential` to undertand the differences between ***formal and informal*** housing in this area.
    
**We want to understand the living space *(the cubic-meter BVPC value)* each person has in thier home**
</div>

In [34]:
formal = gdf_pop[gdf_pop["attr_building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments'])].copy()
f_pop = int(formal['pop'].sum())

informal = gdf_pop[gdf_pop["attr_building"].isin(['residential', 'cabin'])].copy()
inf_pop = int(informal['pop'].sum())

#- student
stu = gdf_pop[gdf_pop["attr_building"].isin(['student', 'dormitory'])].copy()
stu_pop = int(stu['pop'].sum())

bvpc_formal = round(formal['volume_m3'].sum() / formal['pop'].sum(), 3)
bvpc_informal = round(informal['volume_m3'].sum() / informal['pop'].sum() if informal['pop'].sum() != 0 else 0, 3)
bvpc_stu = round(stu['volume_m3'].sum() / stu['pop'].sum() if stu['pop'].sum() != 0 else 0, 3)

print('FORMAL: Population: ', f_pop, ' with Building Volume Per Capita (BVPC):', bvpc_formal)
print('')
print('STUDENT RESIDENCE: Population: ', stu_pop, ' with Building Volume Per Capita (BVPC):', bvpc_stu)
print('')
print('INFORMAL: Population: ', inf_pop, ' with Building Volume Per Capita (BVPC)', bvpc_informal)

FORMAL: Population:  1180  with Building Volume Per Capita (BVPC): 222.166

STUDENT RESIDENCE: Population:  0  with Building Volume Per Capita (BVPC): 0

INFORMAL: Population:  0  with Building Volume Per Capita (BVPC) 0


## 2. Further examples of Spatial Data Science *(renewable energy)*:

<div class="alert alert-block alert-warning"><b></b>

**Let's attempt to understand the % of homes and population served with renewable energy.**
</div>
    
[**SDG**](https://sdgs.un.org/goals) indicators are typically calculated at **region and national scales**.  
Here, because we are working with highly detailed, local data, we can explore what a [**Tier 3 local indicator**](https://unstats.un.org/sdgs/metadata/) might look like at a ***neighbourhood level***.
<br>

In this section 3. we evaluate [**SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all**](https://sdgs.un.org/goals/goal7) at a community level and estimate the **proportion of residential units and population** that have **direct access to on-site renewable energy infrastructure** *--rooftop photovoltaic panels (PV) and solar water heaters (SWH)*.

- Percentage of **households** served by rooftop renewable energy  
- Percentage of the **population** served by rooftop renewable energy
- And then we go even further to estimate the **Annual Solar Potential in MWh** *(theoretical maximum electricity)* that homes can harvest from the sun over the course of one year.

In [35]:
#- harvest the solar renewable from the cityJSON

# Create an empty set to hold unique building IDs with solar infrastructure
solar_building_methods = {}
solar_building_ids = set()

# Access the raw CityObjects dictionary
city_objects = cm.j.get('CityObjects', {})

for obj_id, obj_val in city_objects.items():
    # Target the standalone Energy Extensions directly
    if obj_val.get('type') == '+Energy-PVSystem':
        solar_method = obj_val.get('attributes', {}).get('method', 'unknown')
        # Level 1 Parent: The BuildingInstallation ('Solar_...')
        inst_parent_list = obj_val.get('parents', [])
        if inst_parent_list:
            inst_id = inst_parent_list[0]
            inst_obj = city_objects.get(inst_id, {})
            
            # Level 2 Parent: The real building asset key ('osm_...')
            building_parent_list = inst_obj.get('parents', [])
            if building_parent_list:
                building_id = building_parent_list[0]
                solar_building_ids.add(building_id)
                solar_building_methods[building_id] = solar_method

print(f"Found {len(solar_building_ids)} residential assets with on-site solar.")

Found 30 residential assets with on-site solar.


In [36]:
#- connect the solar renewable to the buildings and population

# 1. Flag each row in your master dataframe if its building_id has solar attached
gdf_pop['has_solar'] = gdf_pop['building_id'].isin(solar_building_ids)

gdf_pop['solar_method'] = gdf_pop['building_id'].map(solar_building_methods).fillna('none')

In [37]:
# 2. Filter down to your residential housing | buildings people live in. we include garage.
formal_homes = gdf_pop[gdf_pop["attr_building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments', 'garage'])].copy()

<div class="alert alert-block alert-success"><b>Percentage of households served by rooftop renewable energy</b></div>

$$ \text{\% homes with renewable energy} = \frac{\text{Number of dwellings with mapped solar PV and SWH}}{\text{Total number of dwellings}} \times 100 $$

In [38]:
# 3. Count total formal dwellings vs those with solar assets
total_formal_dwellings = len(formal_homes)
dwellings_with_solar = formal_homes['has_solar'].sum()

# 4. Compute the final percentage metric scaled out of 100
if total_formal_dwellings > 0:
    pct_homes_solar = round((dwellings_with_solar / total_formal_dwellings) * 100, 3)
else:
    pct_homes_solar = 0.0

print('Percentage of households served by rooftop renewable energy:')
print(f"{pct_homes_solar}%")

Percentage of households served by rooftop renewable energy:
9.603%


<div style="text-align: left;"> 
<small> <b>NB:</b> this number includes the <a href="https://wiki.openstreetmap.org/wiki/Tag:building%3Dgarage">OpenStreetMap <code>building=garage</code></a> building type. Go to <code>Cell &plusmn;37</code> (above) to exclude this building type from the estimate.</small>
</div>
<br>
<div class="alert alert-block alert-success"><b>Percentage of population served by rooftop renewable energy</b></div>

$$ \text{\% population with renewable energy} = \frac{\text{Number of residents with mapped solar PV and SWH}}{\text{Estimated population}} \times 100 $$

In [39]:
# 3. Sum up the residents inside buildings with solar vs total community population
residents_with_solar = int(gdf_pop[gdf_pop['has_solar'] == True]['pop'].sum())
total_estimated_population = int(gdf_pop['pop'].sum())

# 4. Compute the final percentage metric scaled out of 100
if total_estimated_population > 0:
    pct_pop_solar = round((residents_with_solar / total_estimated_population) * 100, 3)
else:
    pct_pop_solar = 0.0

print('Percentage of population served by rooftop renewable energy:')
print(f"{pct_pop_solar}%")

Percentage of population served by rooftop renewable energy:
9.492%


In [40]:
gdf_pop.columns

Index(['building_id', 'lod', 'surface_type', '3d_area', 'tilt_deg',
       'aspect_deg', 'mean_z_absolute', 'attr_osm_id', 'attr_address',
       'attr_building', 'attr_building:levels', 'attr_operator',
       'attr_building_height', 'attr_roof_height', 'attr_ground_height',
       'attr_plus_code', 'attr_footprint', 'geometry', 'pop',
       'total_roof_area', 'ground_footprint', 'volume_m3', 'bvpc', 'has_solar',
       'solar_method'],
      dtype='object')

In [41]:
# number of solar renewable on HOMES
gdf_pop['solar_method'].explode().value_counts()

solar_method
none            278
photovoltaic     20
thermal          10
Name: count, dtype: int64

### 2. ii) Solar potential (MWh)

<div class="alert alert-block alert-warning"><b></b>

In this section, we attempt to understand how much ***'fuel'*** a rooftop can get from the sun.
</div>
<div class="alert alert-block alert-info"><b></b> 
    
To do your own community **PLEASE SUPPLY YOUR OWN** `POWER_Regional_Monthly_1984_2025.csv`; an extract of the `ALLSKY_SFC_SW_DWN` (All Sky Surface Shortwave Downward Irradiance) dataset.

These can be harvested from the [**NASA data access viewer**](https://power.larc.nasa.gov/data-access-viewer/) --settings: Regional, User Community: Renewable, Temporal Level: Monthly and Annually, Select Location: draw a rectangle (~state / province. NOT country), Time Extent: 1984-2025, Parameter: All Sky Surface Shortwave Downward Irradiance), Format: csv.**  

</div>

Here we interogate the [POWER Monthly Radiation](https://nasa.maps.arcgis.com/home/item.html?id=f8b59c55f66c47bba486354a122a5489) dataset. This is a global dataset that uses [NASA satellite observations and weather models](https://power.larc.nasa.gov) to tell us exactly how much solar radiation hits a specific location on Earth.

What we are harvesting:

> **Parameter:** `ALLSKY_SFC_SW_DWN` (Global Horizontal Irradiance): a 30-year historical average of solar radiation. This ensures our communities solar potential is based on ***long-term climate trends*** rather than a single year of weather.
>
> **Source:** [POWER Monthly Radiation](https://nasa.maps.arcgis.com/home/item.html?id=f8b59c55f66c47bba486354a122a5489) via the [**NASA data access viewer**](https://power.larc.nasa.gov/data-access-viewer/)
>   
> **Goal:**
> To calculate the Annual Total GHI (kWh/m2/year). This value tells us the cumulative **'solar pressure'** hitting our rooftops over an entire year, which we then use to calculate **how many Megawatt-hours (MWh) of clean electricity our neighborhood can generate**.

In [42]:
#- NASA POWER monthly radiation
file_path = './data/POWER_Regional_Monthly_1984_2025.csv'

In [43]:
#- we need the center of the area in wgs84

#- Pull the native bounding box from the cityjson metadata object
# bbox is structured as: [minx, miny, minz, maxx, maxy, maxz]
bbox = cm.j.get('bbox')

if bbox:
    # compute the native midpoint in your local system (EPSG:32734)
    native_center_x = (bbox[0] + bbox[3]) / 2.0
    native_center_y = (bbox[1] + bbox[4]) / 2.0
else:
    # fallback to the transform translation anchor point if bbox isn't present
    translate = cm.j.get('transform', {}).get('translate', [0, 0, 0])
    native_center_x, native_center_y = translate[0], translate[1]

#- set up the coordinate transform pipeline using GDAL/OSR
source_crs = osr.SpatialReference()
source_crs.ImportFromEPSG(int(crs[7:]))       # cityJSON crs

target_crs = osr.SpatialReference()
target_crs.ImportFromEPSG(4326)               # Target geographic WGS84 degrees projection

# Add this line right here to force (Lon, Lat) order:
target_crs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
#- ensure modern axis mapping compatibility (Lon, Lat order)
transform = osr.CoordinateTransformation(source_crs, target_crs)

#- execute transformation: returns (longitude, latitude, altitude)
target_lon, target_lat, _ = transform.TransformPoint(native_center_x, native_center_y)
print(target_lon, target_lat)

18.45317128549845 -33.937448875290684


In [44]:
#- load the NASA POWER dataset (skipping header lines as per your configuration)
df = pd.read_csv(file_path, skiprows=9)
df.columns = [c.strip() for c in df.columns]

# Identify the nearest grid point using the transformed degrees
closest_lat = min(df['LAT'].unique(), key=lambda x: abs(x - target_lat))
closest_lon = min(df['LON'].unique(), key=lambda x: abs(x - target_lon))

#print(f"Nearest NASA Power Grid Intersect Matched: LAT {closest_lat}, LON {closest_lon}")

# Filter for the site and exclude missing data indicator lines (-999)
site_df = df[(df['LAT'] == closest_lat) & (df['LON'] == closest_lon)].copy()
site_df = site_df[site_df['ANN'] != -999]

# Harvest the long-term annual climate baseline scaled for complete years
annual_avg_daily = site_df['ANN'].mean()
annual_ghi_kwh_m2 = annual_avg_daily * 365.25

print(f"Long-term Daily Mean GHI: {annual_avg_daily:.3f} kWh/m²/day")
print(f"Annual Cumulative GHI Baseline: {annual_ghi_kwh_m2:.2f} kWh/m²/year")

Long-term Daily Mean GHI: 5.526 kWh/m²/day
Annual Cumulative GHI Baseline: 2018.34 kWh/m²/year


<div class="alert alert-block alert-success"><b>Annual Solar Potential (MWh)</b></div>

We use a simplified formula to provide a clear baseline that accomadates both LoD1 and LoD2 city models

$$
\text{Potential (MWh)} = \frac{(\text{Surface Area} \times \text{utilization factor} \times \text{Solar inclination factor}) \times \text{GHI}_{\text{annual}} \times 0.2}{1000}
$$


<div class="alert alert-block alert-warning"><b></b>

<sub>***Theoretical Framework:** The annual energy output of a photovoltaic system (E) is determined by the product of the total solar resource (GHI), the active area of the array (A), and the system's overall efficiency (η), adjusted by a Performance Ratio (PR) to account for real-world losses. — based on [NREL (2022)](https://joint-research-centre.ec.europa.eu/photovoltaic-geographical-information-system-pvgis_en) & [IEC 61724-1](https://webstore.iec.ch/en/publication/65561). We then adapt this formula and represents a combined value of 25% nominal panel efficiency and a 0.80 Performance Ratio with a **single 0.20 system efficiency value and account for the ORIENTATION of the roof surface**.*</sub> 
</div>

<div class="alert alert-block alert-info"><b></b> 
    
**2. ii) a. We filter the dataset based on the orientation of the roof surface**

</div>

- we need the hemisphere to understand where the sun will be shining from; an azimuth window

In [45]:
def get_solar_azimuth_window(target_lat, target_lon, buffer_deg=67.5):
    """
    Dynamically computes the ideal sun-facing azimuth window based on location.
    Accepts a buffer_deg to define the angular width on either side of the optimum.
    """
    # Southern Hemisphere: Optimal direction is North (0 degrees)
    if target_lat < 0:
        optimal_azimuth = 0.0
        min_azimuth = (optimal_azimuth - buffer_deg) % 360
        max_azimuth = (optimal_azimuth + buffer_deg) % 360
        hemisphere = "Southern (Optimizing North)"
    
    # Northern Hemisphere: Optimal direction is South (180 degrees)
    else:
        optimal_azimuth = 180.0
        min_azimuth = (optimal_azimuth - buffer_deg)
        max_azimuth = (optimal_azimuth + buffer_deg)
        hemisphere = "Northern (Optimizing South)"
        
    return min_azimuth, max_azimuth, hemisphere

#- dynamically find the window using your parsed coordinates --- (Assumes GDAL pipeline unpacked target_lat correctly after axis handling)
min_sun_azimuth, max_sun_azimuth, hemi_info = get_solar_azimuth_window(target_lat, target_lon, buffer_deg=67.5)

print(f"Detected Location Hemisphere: {hemi_info}")
print(f"Dynamic Azimuth Window: {min_sun_azimuth}° to {max_sun_azimuth}°\n")

Detected Location Hemisphere: Southern (Optimizing North)
Dynamic Azimuth Window: 292.5° to 67.5°



In [46]:
# Create a copy of your roof surfaces dataframe to evaluate solar performance

#- define strict residential allowed list
allowed_residential = ['house', 'semidetached_house', 'terrace', 'terraced', 'apartments', 'residential', 'dormitory', 'cabin', 'student', 'garage']

#- filter the master geodataframe strictly for these types first (ensure 'attr_building' matches your column name exactly)
residential_gdf = gdf[gdf['attr_building'].isin(allowed_residential)].copy()
residential_gdf.head(2)

,building_id,lod,surface_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry,face_signed_volume
12,osm_739615941,2.0,RoofSurface,6.031476,0.0,0.0,100.2,739615941,10 Rhodes Avenue University Estate Cape Town,house,2.0,NaN,6.9,101.0,93.3,4FRW3C6X+WRX,"[[[264275.999, 6241813.835], [264270.087, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...,201.445458
13,osm_739615941,2.0,RoofSurface,5.959110,0.0,0.0,100.2,739615941,10 Rhodes Avenue University Estate Cape Town,house,2.0,NaN,6.9,101.0,93.3,4FRW3C6X+WRX,"[[[264275.999, 6241813.835], [264270.087, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...,199.035646


In [47]:
#- isolate explicit LoD2 roofs and flat LoD1 roofs from this residential-only subset

is_explicit_roof = residential_gdf['surface_type'] == 'RoofSurface'
roof_analysis = residential_gdf[is_explicit_roof].copy()

len(roof_analysis)

1592

- LoD2 roof surfaces facing the sun

In [48]:
#- target_lon
if min_sun_azimuth > max_sun_azimuth:
    sun_facing_filter = (roof_analysis['aspect_deg'] >= min_sun_azimuth) | (roof_analysis['aspect_deg'] <= max_sun_azimuth)
else:
    # Northern Hemisphere standard range (e.g., 112.5° to 247.5°)
    sun_facing_filter = (roof_analysis['aspect_deg'] >= min_sun_azimuth) & (roof_analysis['aspect_deg'] <= max_sun_azimuth)

# Apply dynamic mask
solar_viable_roofs = roof_analysis[sun_facing_filter].copy()

print(f"Starting total roof facets: {len(roof_analysis)}")
print(f"Facets orienting towards the sun: {len(solar_viable_roofs)}")

Starting total roof facets: 1592
Facets orienting towards the sun: 801


<div class="alert alert-block alert-info"><b></b> 
    
**2. ii) b. We further refine the LoD2 dataset based on the inclination** (tilt) **of the roof surfaces**

</div>

In [49]:
#- inclination / tilt
def calculate_inclination_efficiency(tilt, target_lat):
    """
    Calculates the vertical scaling factor based strictly on roof pitch.
    Optimizes for an inclination matching the absolute value of the local latitude.
    """
    tilt_rad = np.radians(tilt)
    lat_rad = np.radians(abs(target_lat)) # Cape Town latitude magnitude (~33.93°)
    
    # Cosine of the difference between the actual roof tilt and the ideal latitude tilt
    inclination_factor = np.cos(tilt_rad - lat_rad)
    
    return float(inclination_factor)

# Apply the pure vertical tilt efficiency factor to your filtered sun-facing dataframe
solar_viable_roofs['tilt_efficiency_factor'] = solar_viable_roofs.apply(
    lambda row: calculate_inclination_efficiency(row['tilt_deg'], target_lat), axis=1
)

In [50]:
# Focus strictly on Structural Engineering Tilt Filtering
MIN_VIABLE_TILT = 0.0    # include flat roofs
MAX_VIABLE_TILT = 40.0   # Exclude steep pitches that increase structural wind loads

# Apply the strict structural tilt filter mask to your orientation-filtered data
final_solar_analysis = solar_viable_roofs[
    (solar_viable_roofs['tilt_deg'] <= MAX_VIABLE_TILT)
].copy()

print(f"Total sun-facing roof facets: {len(solar_viable_roofs)}")
print(f"Facets remaining after structural tilt filtering: {len(final_solar_analysis)}")
print(f"Facets dropped due to unviable inclination: {len(solar_viable_roofs) - len(final_solar_analysis)}")

Total sun-facing roof facets: 801
Facets remaining after structural tilt filtering: 801
Facets dropped due to unviable inclination: 0


<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    
**Fill in the LoD1 `utilization_factor` below** 
</div>

As a **'rule-of-thumb'** a community with traditional gabled houses: `utilization_factor = 0.4` *(less than half)*, while a high-density suburb with flat-roofed apartments: `utilization_factor = 0.6`

In [51]:
# on average, how much of the roof faces the sun? | is useable. Adjust based on roof types
LoD1utilization_factor = 0.4

<div class="alert alert-block alert-warning"><b></b>

We can hardcode the LoD2 `utilization_factor` at say 0.65; because with LoD2 we're only considering space for setbacks, walkways, etc.
</div>

In [52]:
#- dynamic utilization rules based on the LoD

final_solar_analysis['roof_utilization'] = np.where(
    final_solar_analysis['lod'].astype(str).str.startswith('1'),
    LoD1utilization_factor,                  # LoD1: unless flat; most of the roof will NOT face then sun. typically less than half. 0.4
    0.65                                     # LoD2:  
)

#### execute the solar potential formula

In [53]:
#- execute Annual Solar Potential (MWh) formula 

SYSTEM_EFFICIENCY = 0.20  # Combined 20% system efficiency factor

final_solar_analysis['solar_potential_mwh'] = ((final_solar_analysis['3d_area'] *               # the area of the roof surface
                                                final_solar_analysis['roof_utilization'] *      # LoD2 facing the sun * 0.65 util_fac | LoD1 * 0.4 util_fac
                                                final_solar_analysis['tilt_efficiency_factor']) # exclude steep pitched roof surfaces 
                                               * annual_ghi_kwh_m2                              # ghi from NASA
                                               * SYSTEM_EFFICIENCY                              # system efficiency
                                              ) / 1000.0


In [54]:
final_solar_analysis.lod.unique()

array(['2.0', '1.0'], dtype=object)

In [55]:
#- map the results back to the buildings.gdf with the population and bvpc metrics

#- Aggregate the clean filtered facet yields back up to the master building entities
building_potentials = final_solar_analysis.groupby('building_id')['solar_potential_mwh'].sum().to_dict()

#- Map results directly back into your main demographic population dataframe
# Buildings that had no viable roof surfaces after filtering will safely default to 0.0 MWh
gdf_pop['solar_potential_mwh'] = gdf_pop['building_id'].map(building_potentials).fillna(0.0)
gdf_pop.head(2)

,building_id,lod,surface_type,3d_area,tilt_deg,aspect_deg,mean_z_absolute,attr_osm_id,attr_address,attr_building,...,attr_footprint,geometry,pop,total_roof_area,ground_footprint,volume_m3,bvpc,has_solar,solar_method,solar_potential_mwh
0,osm_13705114,1.0,WallSurface,536.217387,90.0,204.05,73.05,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,...,"[[[264304.86, 6241972.753], [264318.134, 62420...",POLYGON Z ((264304.860314 6241972.752832 69.60...,NaN,0.000000,0.0,16778.099256,NaN,False,none,0.000000
12,osm_739615941,2.0,RoofSurface,6.031476,0.0,0.00,100.20,739615941,10 Rhodes Avenue University Estate Cape Town,house,...,"[[[264275.999, 6241813.835], [264270.087, 6241...",POLYGON Z ((264270.234314 6241816.283832 100.2...,4.0,6.031476,0.0,1013.307644,253.326911,True,photovoltaic,6.798877


In [56]:
#- look
optimized_mean_mwh = gdf_pop['solar_potential_mwh'].mean()
print("\033[1mThe average solar potential, per home\033[0m, for", jparams['FocusArea'], "is:",  round(optimized_mean_mwh, 2), "MWh/year")

The average solar potential, per home, for University Estate is: 18.28 MWh/year


<div style="text-align: left;"> 
<small> <b>NB:</b> these number includes the <a href="https://wiki.openstreetmap.org/wiki/Tag:building%3Dgarage">OpenStreetMap <code>building=garage</code></a> building type. Go to <code>Cell &plusmn;46</code> to exclude this building type from the estimate.</small>
</div>
<br>

<div class="alert alert-block alert-info"><b></b>
   
What does this **MWh/year** value mean?
<br>
    
To put the value in context, **15 MWh/year:**  
- is enough to provide **100% of the electricity** for 4 to 5 average UK homes *(which use ~3.4 MWh each)* or 1.5 average US homes *(~10.7 MWh each)*
- is enough power to drive an Electric Vehicle for **75,000 kilometers** *--that’s almost two full trips around the Earth*.
- roughly saves **10 metric tons of Carbon Dioxide** from entering the atmosphere.
    
</div>

<div class="alert alert-block alert-warning"><b></b>
And if you want to establish a micro-grid for the community and share resources
</div>

In [57]:
#- look
optimized_total_mwh = gdf_pop['solar_potential_mwh'].sum()
print("\033[1mTotal Study Area Rooftop Solar Generation Potential\033[0m, for", jparams['FocusArea'], "is:",  round(optimized_total_mwh, 2), "MWh/year")

Total Study Area Rooftop Solar Generation Potential, for University Estate is: 5629.63 MWh/year


<div style="text-align: left;"> 
<small> <b>NB:</b> these number includes the <a href="https://wiki.openstreetmap.org/wiki/Tag:building%3Dgarage">OpenStreetMap <code>building=garage</code></a> building type. Go to <code>Cell &plusmn;46</code> to exclude this building type from the estimate.</small>
</div>

<div class="alert alert-block alert-success"><b>Community Micro-grid Sizing Metrics</b></div>

In [58]:
# --- micro-grid community sizing metrics ---

#- total available roof real estate for community sharing
total_sharing_area = (final_solar_analysis['3d_area'] *  final_solar_analysis['roof_utilization']).sum()

#- count of viable producer nodes (homes that can contribute surplus to the grid)
producer_nodes_count = (gdf_pop['solar_potential_mwh'] > 0).sum()
total_nodes = len(gdf_pop)

print(f"\n\033[1mMicro-grid Resource Summary for {jparams['FocusArea']}:\033[0m")
print(f"* Total collective solar collection footprint: {total_sharing_area:,.2f} m²")
print(f"* Grid Participation: {producer_nodes_count} out of {total_nodes} homes have viable sun-facing surfaces to act as generation nodes.")


Micro-grid Resource Summary for University Estate:
* Total collective solar collection footprint: 16,220.87 m²
* Grid Participation: 286 out of 308 homes have viable sun-facing surfaces to act as generation nodes.


<div class="alert alert-block alert-danger"><b>Sanity Check!</b>
<br>

- In <code>Cell &plusmn;37</code> we excluded non-residential building types `=office, commercial, retail, warehouse, industrial, etc.` from the analysis.
    
- Cape Town typically yields ~1.6–1.7 MWh **per installed kWp per year**; the higher per-household values reported here reflect **rooftop potential derived from available area**, not a 1 kWp system.

    A 1 kWp solar PV system requires approximately 5–8 m² of panel area (e.g. panels of roughly 1 m × 1.7 m, depending on technology).

    We are NOT asking: How much energy (MWh/year) would a single 1 kWp PV system generate on a roof?  
    We are asking: **How much energy (MWh/year) could these roofs harvest, given their available area, orientation and inclination**/ tilt. 

</div>